In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/annotated/checkpoints/post_cell_16.pickle

In [ ]:
%%RecordEvent
%%time
### cell 17 ###

# Optimized Cell 17 for cudf.pandas
def bar_chart_multiple_years_26(df, x_column, y_column, title, y_axis_title):
    # GPU‐accelerated write to CSV
    df.to_csv(
        f'/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results'
        f'/kaggle/working/individual_charts/data/{title}.csv',
        index=True
    )

def grab_subset_of_data_26(df, question_of_interest):
    # select columns containing the question and rename in one pass
    sub = df.filter(like=question_of_interest)
    sub = sub.rename(columns=lambda col: col.split('-')[-1].lstrip())
    # drop rows that are all-null across those columns
    return sub.dropna(how='all')

def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
    question_of_interest, include_2017=False
):
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs   = [responses_df_2018, responses_df_2019, responses_df_2020, responses_df_2021, responses_df_2022]
    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)
    # build list of GPU dataframes with a year column
    processed = [
        grab_subset_of_data_26(df, question_of_interest).assign(year=yr)
        for df, yr in zip(dfs, years)
    ]
    combined = pd.concat(processed, ignore_index=True)
    counts   = combined.groupby('year').count().reset_index()
    return combined, counts

def convert_df_of_counts_to_percentages_26(df, counts):
    # vectorized percentage calculation per year
    totals = df['year'].value_counts().sort_index()
    pct    = (
        counts
        .set_index('year')
        .astype('float64')
        .div(totals, axis=0)
        * 100
    )
    return pct.reset_index()

# normalize some 2018/2019 values and column names on GPU
responses_df_2018.columns = (
    responses_df_2018.columns
    .str.replace(question_of_interest_alternate, question_of_interest)
)
responses_df_2018 = (
    responses_df_2018
    .replace('Kaggle Learn', 'Kaggle Learn Courses')
    .replace('Fast.AI', 'Fast.ai')
    .replace(
        'Online University Courses',
        'University Courses (resulting in a university degree)'
    )
    .rename(columns=lambda col: col
        .replace('Kaggle Learn', 'Kaggle Learn Courses')
        .replace('Fast.AI', 'Fast.ai')
        .replace(
            'Online University Courses',
            'University Courses (resulting in a university degree)'
        )
    )
)
responses_df_2019 = (
    responses_df_2019
    .replace(
        'Kaggle Courses (i.e. Kaggle Learn)',
        'Kaggle Learn Courses'
    )
    .rename(columns=lambda col: col
        .replace(
            'Kaggle Courses (i.e. Kaggle Learn)',
            'Kaggle Learn Courses'
        )
    )
)

# user inputs / active vars
question_of_interest = 'On which platforms have you begun or completed data science courses?'
title_for_chart       = 'Most popular learning platforms (2018-2022)'
# title_for_y_axis remains unchanged

# combine, convert, reshape
combined_df, counts_df = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_26(
        question_of_interest
    )
)
percentages_df = convert_df_of_counts_to_percentages_26(combined_df, counts_df)
# clean up the long column label
percentages_df.columns = percentages_df.columns.str.replace(
    '(resulting in a university degree)',
    '',
    regex=False
)
# select and order columns for plotting
cols_order = [
    'year', 'Coursera', 'University Courses ', 'Kaggle Learn Courses',
    'Udemy', 'Udacity', 'DataCamp', 'edX', 'Fast.ai', 'None', 'Other'
]
df = (
    percentages_df.loc[:, cols_order]
    .melt(id_vars='year', value_vars=cols_order[1:])
    .sort_values(['year', 'value'], ascending=True)
    .rename(columns={'variable': ''})
)

bar_chart_multiple_years_26(df, '', 'value', title_for_chart, title_for_y_axis)


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high/checkpoints/post_cell_17_try_7.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_17_try_7.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[17], f)


In [ ]:
opt_output = Out.get(4)